# 🛡️ Rakshak AI Engine — Complete Training Pipeline

**Multi-Model Deep Learning System for Railway Predictive Maintenance**

This notebook trains **4 models** on the Rakshak massive dataset (250M sensor readings, 50 scenarios):

| Model | Architecture | Purpose | Target |
|-------|-------------|---------|--------|
| **VAE** | Conv1D Autoencoder | Anomaly Detection (Tier 3) | F1 ≥ 0.96 |
| **Failure Predictor** | TCN + Transformer + BiLSTM | 1h/6h/24h Failure Prediction | AUROC ≥ 0.95 |
| **Fault Classifier** | ResNet-1D | Root Cause Identification | Top-1 ≥ 0.85 |
| **Ensemble** | IsoForest + GBM | 3-Tier Anomaly Meta-Classifier | FPR < 4% |

---
**Instructions:**
1. Upload `ai_engin/` folder to Google Drive
2. Set GPU runtime: `Runtime → Change runtime type → T4 GPU`
3. Run all cells in order
4. Download exported models from `ai_engin/trained_models/`

## 1. Setup & Mount Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# ══════════════════════════════════════════════════
# ⚙️ CONFIGURE THIS — path to ai_engin on your Drive
# ══════════════════════════════════════════════════
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/ai_engin'
# ══════════════════════════════════════════════════

assert os.path.isdir(DRIVE_PROJECT_DIR), f'ai_engin folder not found at {DRIVE_PROJECT_DIR}'
print(f'✅ Found ai_engin at: {DRIVE_PROJECT_DIR}')
print(f'Contents: {os.listdir(DRIVE_PROJECT_DIR)}')

In [ ]:
# Install dependencies
!pip install -q lightgbm xgboost pyarrow

# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# Add project to Python path
import sys
COLAB_TRAINING_DIR = os.path.join(DRIVE_PROJECT_DIR, 'colab_training')
sys.path.insert(0, COLAB_TRAINING_DIR)

# Create working directories
WORK_DIR = '/content/rakshak_training'
os.makedirs(os.path.join(WORK_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, 'logs'), exist_ok=True)

# Override config paths for Colab
import config
config.DATASET_ZIP_PATH = os.path.join(DRIVE_PROJECT_DIR, 'rakshak_massive_dataset.zip')
config.CHECKPOINT_DIR = os.path.join(WORK_DIR, 'checkpoints')
config.EXPORT_DIR = os.path.join(DRIVE_PROJECT_DIR, 'trained_models')
config.LOG_DIR = os.path.join(WORK_DIR, 'logs')

# Setup logging
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(name)s] %(levelname)s: %(message)s',
    datefmt='%H:%M:%S',
)

print(f'✅ Dataset: {config.DATASET_ZIP_PATH}')
print(f'✅ Checkpoints: {config.CHECKPOINT_DIR}')
print(f'✅ Export: {config.EXPORT_DIR}')

## 2. Load & Explore Dataset

In [ ]:
from data.data_loader import ParquetStreamLoader, split_scenarios_by_type

# Initialize loader
loader = ParquetStreamLoader(config.DATASET_ZIP_PATH)

print(f'Total parquet files: {loader.num_files}')
print(f'Total scenarios: {loader.num_scenarios}')

# Show scenario summary
summary = loader.get_scenario_summary()
print(f'\nScenarios by fault type:')
print(summary.groupby('fault_type')['num_chunks'].sum().sort_values(ascending=False).to_string())
print(f'\nNormal vs Anomaly: {summary["is_normal"].value_counts().to_dict()}')

In [ ]:
# Split scenarios into train/val/test
train_files, val_files, test_files = split_scenarios_by_type(loader)
print(f'Train files: {len(train_files)}')
print(f'Val files: {len(val_files)}')
print(f'Test files: {len(test_files)}')

## 3. Prepare Data (Streaming)

In [ ]:
import numpy as np
from data.feature_engineer import engineer_features, get_feature_columns
from data.preprocessing import (
    FeatureNormalizer, FaultLabelEncoder,
    preprocess_chunk_for_vae,
    preprocess_chunk_for_classifier,
    create_windows_with_labels,
    generate_failure_windows,
    compute_fault_class_weights,
)
from tqdm import tqdm

# We'll collect windows from streaming chunks
vae_train_windows, vae_train_labels = [], []
vae_val_windows, vae_val_labels = [], []

fail_train_windows, fail_train_labels = [], {}
fail_val_windows, fail_val_labels = [], {}

cls_train_windows, cls_train_labels = [], []
cls_val_windows, cls_val_labels = [], []

# Feature setup
feature_columns = get_feature_columns()
fault_encoder = FaultLabelEncoder()
vae_normalizer = FeatureNormalizer()
fail_normalizer = FeatureNormalizer()
cls_normalizer = FeatureNormalizer()

print(f'Engineered features: {len(feature_columns)}')
print(f'Fault classes: {fault_encoder.num_classes}')
print(f'Window size: {config.DATA_CONFIG.window_size}')

In [ ]:
# ═══════════════════════════════════════════════
# Stream training data and build windowed datasets
# ═══════════════════════════════════════════════

print('Processing TRAINING data...')
first_chunk = True

for chunk_df in tqdm(loader.stream(
    files_per_chunk=10,
    max_rows_per_file=config.DATA_CONFIG.max_rows_per_file,
    file_list=train_files,
), total=len(train_files)//10 + 1, desc='Train chunks'):

    # Engineer features
    chunk_eng = engineer_features(chunk_df)

    # --- VAE (uses raw features only) ---
    w, l, vae_normalizer = preprocess_chunk_for_vae(
        chunk_df, vae_normalizer, fit=first_chunk
    )
    if len(w) > 0:
        vae_train_windows.append(w)
        vae_train_labels.append(l)

    # --- Failure Predictor (uses engineered features) ---
    if first_chunk:
        fail_normalizer.fit(chunk_eng, feature_columns)
    chunk_norm = fail_normalizer.transform(chunk_eng)
    features = chunk_norm[feature_columns].values.astype(np.float32)
    anomaly_labels = chunk_df[config.LABEL_COL].values.astype(np.float32)
    fw, fl = generate_failure_windows(features, anomaly_labels)
    if len(fw) > 0:
        fail_train_windows.append(fw)
        for h, labels in fl.items():
            fail_train_labels.setdefault(h, []).append(labels)

    # --- Fault Classifier (anomalous samples only) ---
    w, l, cls_normalizer = preprocess_chunk_for_classifier(
        chunk_eng, feature_columns + ['fault_type'],
        fault_encoder, cls_normalizer, fit=first_chunk,
    )
    if len(w) > 0:
        cls_train_windows.append(w)
        cls_train_labels.append(l)

    first_chunk = False

# Concatenate all chunks
vae_train_X = np.concatenate(vae_train_windows)
vae_train_y = np.concatenate(vae_train_labels)
fail_train_X = np.concatenate(fail_train_windows)
fail_train_y = {h: np.concatenate(lbls) for h, lbls in fail_train_labels.items()}
cls_train_X = np.concatenate(cls_train_windows) if cls_train_windows else np.empty((0,64,len(feature_columns)))
cls_train_y = np.concatenate(cls_train_labels) if cls_train_labels else np.empty((0,))

print(f'\nDataset shapes:')
print(f'  VAE train: {vae_train_X.shape}, labels: {vae_train_y.shape}')
print(f'  Failure train: {fail_train_X.shape}')
print(f'  Classifier train: {cls_train_X.shape}, labels: {cls_train_y.shape}')

In [ ]:
# ═══════════════════════════════════════════════
# Stream validation data
# ═══════════════════════════════════════════════

print('Processing VALIDATION data...')
for chunk_df in tqdm(loader.stream(
    files_per_chunk=10,
    max_rows_per_file=config.DATA_CONFIG.max_rows_per_file,
    file_list=val_files,
), total=len(val_files)//10 + 1, desc='Val chunks'):

    chunk_eng = engineer_features(chunk_df)

    w, l, _ = preprocess_chunk_for_vae(chunk_df, vae_normalizer, fit=False)
    if len(w) > 0:
        vae_val_windows.append(w)
        vae_val_labels.append(l)

    chunk_norm = fail_normalizer.transform(chunk_eng)
    features = chunk_norm[feature_columns].values.astype(np.float32)
    anomaly_labels = chunk_df[config.LABEL_COL].values.astype(np.float32)
    fw, fl = generate_failure_windows(features, anomaly_labels)
    if len(fw) > 0:
        fail_val_windows.append(fw)
        for h, labels in fl.items():
            fail_val_labels.setdefault(h, []).append(labels)

    w, l, _ = preprocess_chunk_for_classifier(
        chunk_eng, feature_columns + ['fault_type'],
        fault_encoder, cls_normalizer, fit=False,
    )
    if len(w) > 0:
        cls_val_windows.append(w)
        cls_val_labels.append(l)

vae_val_X = np.concatenate(vae_val_windows)
vae_val_y = np.concatenate(vae_val_labels)
fail_val_X = np.concatenate(fail_val_windows)
fail_val_y = {h: np.concatenate(lbls) for h, lbls in fail_val_labels.items()}
cls_val_X = np.concatenate(cls_val_windows) if cls_val_windows else np.empty((0,64,len(feature_columns)))
cls_val_y = np.concatenate(cls_val_labels) if cls_val_labels else np.empty((0,))

print(f'\nValidation shapes:')
print(f'  VAE: {vae_val_X.shape}')
print(f'  Failure: {fail_val_X.shape}')
print(f'  Classifier: {cls_val_X.shape}')

## 4. Train Model 1 — VAE Anomaly Detector

In [ ]:
from data.dataset import VAEDataset, create_dataloaders, SensorAugmentation
from training.train_vae import VAETrainer
from evaluation.visualization import plot_vae_history, plot_anomaly_score_distribution

# Create datasets
train_aug = SensorAugmentation(noise_std=0.03)
vae_train_ds = VAEDataset(vae_train_X, vae_train_y, is_training=True, transform=train_aug)
vae_val_ds = VAEDataset(vae_val_X, vae_val_y, is_training=False)

vae_loaders = create_dataloaders(
    vae_train_ds, vae_val_ds,
    batch_size=config.VAE_CONFIG.batch_size,
    balance_train=True,
    train_labels=vae_train_y,
)

print(f'VAE Train batches: {len(vae_loaders["train"])}')
print(f'VAE Val batches: {len(vae_loaders["val"])}')

In [ ]:
# Train VAE
vae_trainer = VAETrainer()
vae_history = vae_trainer.train(
    vae_loaders['train'],
    vae_loaders['val'],
    max_epochs=config.VAE_CONFIG.max_epochs,
)

# Compute optimal threshold
vae_threshold = vae_trainer.compute_threshold(vae_loaders['val'])

# Save scaler
import joblib
joblib.dump(vae_normalizer, os.path.join(config.CHECKPOINT_DIR, 'vae_scaler.joblib'))

In [ ]:
# Plot VAE training history
plot_vae_history(vae_history)

## 5. Train Model 2 — Failure Predictor

In [ ]:
from data.dataset import FailurePredictionDataset, create_dataloaders
from training.train_failure import FailurePredictorTrainer
from evaluation.visualization import plot_failure_prediction_horizons

# Create datasets
fail_train_ds = FailurePredictionDataset(fail_train_X, fail_train_y)
fail_val_ds = FailurePredictionDataset(fail_val_X, fail_val_y)

fail_loaders = create_dataloaders(
    fail_train_ds, fail_val_ds,
    batch_size=config.FAILURE_CONFIG.batch_size,
    balance_train=False,
)

print(f'Failure Train batches: {len(fail_loaders["train"])}')
print(f'Failure Val batches: {len(fail_loaders["val"])}')

In [ ]:
# Train Failure Predictor
failure_trainer = FailurePredictorTrainer()
failure_history = failure_trainer.train(
    fail_loaders['train'],
    fail_loaders['val'],
    max_epochs=config.FAILURE_CONFIG.max_epochs,
)

# Save scaler
joblib.dump(fail_normalizer, os.path.join(config.CHECKPOINT_DIR, 'failure_scaler.joblib'))

In [ ]:
plot_failure_prediction_horizons(failure_history)

## 6. Train Model 3 — Fault Classifier

In [ ]:
from data.dataset import FaultClassificationDataset, create_dataloaders
from data.preprocessing import compute_fault_class_weights
from training.train_classifier import FaultClassifierTrainer
from evaluation.visualization import plot_training_history

# Compute class weights
class_weights = compute_fault_class_weights(cls_train_y, fault_encoder.num_classes)
print(f'Class weights: {dict(zip(fault_encoder.classes[:5], class_weights[:5]))}')

# Create datasets
cls_train_ds = FaultClassificationDataset(cls_train_X, cls_train_y, transform=train_aug)
cls_val_ds = FaultClassificationDataset(cls_val_X, cls_val_y)

cls_loaders = create_dataloaders(
    cls_train_ds, cls_val_ds,
    batch_size=config.FAULT_CONFIG.batch_size,
    balance_train=True,
    train_labels=cls_train_y,
)

print(f'Classifier Train batches: {len(cls_loaders["train"])}')
print(f'Classifier Val batches: {len(cls_loaders["val"])}')

In [ ]:
# Train Fault Classifier
classifier_trainer = FaultClassifierTrainer(class_weights=class_weights)
classifier_history = classifier_trainer.train(
    cls_loaders['train'],
    cls_loaders['val'],
    max_epochs=config.FAULT_CONFIG.max_epochs,
)

# Save scaler + encoder
joblib.dump(cls_normalizer, os.path.join(config.CHECKPOINT_DIR, 'classifier_scaler.joblib'))
fault_encoder.save(os.path.join(config.CHECKPOINT_DIR, 'fault_label_encoder.joblib'))

In [ ]:
plot_training_history(classifier_history, title='Fault Classifier')

## 7. Train Model 4 — Ensemble (IsoForest + GBM Meta)

In [ ]:
from training.train_ensemble import EnsembleTrainer

# Load the trained VAE for Tier 3 scores
vae_model = vae_trainer.model

# Train ensemble
ensemble_trainer = EnsembleTrainer(vae_model=vae_model)
ensemble_metrics = ensemble_trainer.train(
    train_windows=vae_train_X,
    train_labels=vae_train_y,
    val_windows=vae_val_X,
    val_labels=vae_val_y,
)

## 8. Full Evaluation

In [ ]:
from evaluation.evaluate_all import run_full_evaluation
from evaluation.visualization import plot_summary_dashboard

# Run evaluation suite
eval_results = run_full_evaluation(
    checkpoint_dir=config.CHECKPOINT_DIR,
    test_loaders={
        'vae': vae_loaders['val'],
        'failure': fail_loaders['val'],
        'classifier': cls_loaders['val'],
    },
    test_windows=vae_val_X,
    test_labels=vae_val_y,
)

# Print target comparison
print('\n' + '='*60)
print('📊 TARGET vs ACHIEVED')
print('='*60)
for name, info in eval_results.get('target_comparison', {}).items():
    print(f"  {name}: {info['status']} (target={info['target']}, actual={info['actual']})")

In [ ]:
# Plot dashboard
all_metrics = {}
for key in ['vae', 'failure_predictor', 'ensemble']:
    if key in eval_results:
        all_metrics.update(eval_results[key] if isinstance(eval_results[key], dict) else {})
if 'fault_classifier' in eval_results:
    all_metrics.update(eval_results['fault_classifier'].get('metrics', {}))

plot_summary_dashboard(all_metrics)

## 9. Export Models

In [ ]:
from export.export_models import export_all_models

export_dir = export_all_models(
    checkpoint_dir=config.CHECKPOINT_DIR,
    export_dir=config.EXPORT_DIR,
    evaluation_metrics=all_metrics,
    anomaly_threshold=vae_threshold,
)

print(f'\n✅ Models exported to: {export_dir}')
print(f'\nExported files:')
for f in sorted(os.listdir(export_dir)):
    size = os.path.getsize(os.path.join(export_dir, f))
    print(f'  {f:45s} {size/1024:.0f} KB')

## 10. Next Steps

1. **Download** the `trained_models/` directory from Google Drive
2. **Place** it in `ai_engin/trained_models/` in your project
3. **Test** the inference pipeline locally:

```python
from ai_engin.inference.pipeline import RakshakInferencePipeline

pipeline = RakshakInferencePipeline(model_dir='ai_engin/trained_models/')

# Process readings
for i in range(64):  # Fill the buffer
    result = pipeline.process_reading(
        ambient_temp=42.5, humidity=22.0,
        vibration_rms=0.85, gauge_width=1676.3,
    )

print(result.to_dict())
```

4. The backend `agents/` module can import and use the pipeline directly.